---
## Part 1: Clustering

In [1]:
from pyspark.sql import SparkSession
from pyspark.mllib.linalg import Vectors
import numpy as np
import random
import time

spark = SparkSession.builder.master("local[*]").appName("Clustering").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")


In [2]:
def sqdist(a, b):
    diff = np.array(a) - np.array(b)
    return float(np.dot(diff, diff))


In [3]:
def readVectorsSeq(filename):
    points = []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            values = [float(x) for x in line.split(",")]
            points.append(Vectors.dense(values))
    return points


In [4]:
def kcenter(P, k):
    centers = [P[0]]
    for _ in range(k - 1):
        farthest = max(P, key=lambda p: min(sqdist(p, c) for c in centers))
        centers.append(farthest)
    return centers


In [5]:
def kmeansPP(P, k):
    centers = [random.choice(P)]
    for _ in range(k - 1):
        distances = [min(sqdist(p, c) for c in centers) for p in P]
        total = sum(distances)
        probabilities = [d / total for d in distances]
        cumulative = 0.0
        r = random.random()
        chosen = P[-1]
        for p, prob in zip(P, probabilities):
            cumulative += prob
            if r <= cumulative:
                chosen = p
                break
        centers.append(chosen)
    return centers


In [6]:
def kmeansObj(P, C):
    total = sum(min(sqdist(p, c) for c in C) for p in P)
    return total / len(P)


In [7]:
DATA_PATH = r"C:\Projects\IITJ\MLDB\Assignment-4\Q1_UCI_Spam_clustering\spambase.data"

P = readVectorsSeq(DATA_PATH)
print(f"Loaded {len(P)} points with {len(P[0])} dimensions")


Loaded 4601 points with 58 dimensions


In [8]:
k  = 10
k1 = 30
random.seed(42)

t0 = time.time()
kcenter_centers = kcenter(P, k)
print(f"[Step 1] kcenter(P, {k}) running time: {time.time() - t0:.3f}s")


[Step 1] kcenter(P, 10) running time: 0.699s


In [9]:
t0 = time.time()
C_kpp = kmeansPP(P, k)
print(f"[Step 2] kmeansPP(P, {k}) running time: {time.time() - t0:.3f}s")
obj_kpp = kmeansObj(P, C_kpp)
print(f"         kmeansObj = {obj_kpp:.6f}")


[Step 2] kmeansPP(P, 10) running time: 0.702s
         kmeansObj = 31251.603643


In [10]:
t0 = time.time()
X = kcenter(P, k1)
C_coreset = kmeansPP(X, k)
print(f"[Step 3] kcenter(P, {k1}) then kmeansPP(X, {k}): {time.time() - t0:.3f}s")
obj_coreset = kmeansObj(P, C_coreset)
print(f"         kmeansObj = {obj_coreset:.6f}")
print()
print(f"Direct kmeans++ objective:  {obj_kpp:.6f}")
print(f"Coreset kmeans++ objective: {obj_coreset:.6f}")


[Step 3] kcenter(P, 30) then kmeansPP(X, 10): 16.642s
         kmeansObj = 818109.067877

Direct kmeans++ objective:  31251.603643
Coreset kmeans++ objective: 818109.067877


---
## Part 2: Web Search – Inverted Index

In [11]:
import re

STOP_WORDS = {
    "a", "an", "the", "they", "these", "this", "for", "is", "are",
    "was", "of", "or", "and", "does", "will", "whose"
}

PUNCTUATION_RE = re.compile(r'[{}\[\]<>=().,;\'"?#!\-:]')

VALID_TOKEN_RE = re.compile(r'^[a-zA-Z][a-zA-Z+#]*$')

SINGULAR_MAP = {
    "stacks": "stack", "structures": "structure", "applications": "application",
    "elements": "element", "items": "item", "functions": "function",
    "engineers": "engineer", "magazines": "magazine",
    "implementations": "implementation", "plates": "plate",
    "things": "thing", "calls": "call", "lists": "list",
    "terms": "term", "pages": "page", "words": "word",
    "collections": "collection", "computers": "computer",
    "integers": "integer", "documents": "document", "entries": "entry",
}

def normalize(w):
    w = w.lower()
    return SINGULAR_MAP.get(w, w)

def tokenize(text):
    text = PUNCTUATION_RE.sub(" ", text)
    result = []
    position = 0
    for token in text.split():
        position += 1
        if token.lower() in STOP_WORDS:
            continue
        if not VALID_TOKEN_RE.match(token):
            continue
        result.append((normalize(token), position))
    return result


In [12]:
class Position:
    def __init__(self, page_entry, word_index):
        self._page = page_entry
        self._index = word_index

    def getPageEntry(self):
        return self._page

    def getWordIndex(self):
        return self._index


class WordEntry:
    def __init__(self, word):
        self.word = word
        self._positions = []

    def addPosition(self, position):
        self._positions.append(position)

    def addPositions(self, positions):
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self):
        return list(self._positions)

    def getTermFrequency(self, page_name):
        count = sum(1 for p in self._positions if p.getPageEntry().pageName == page_name)
        total = len(self._positions)
        return count / total if total > 0 else 0.0


In [13]:
class MyHashTable:
    def __init__(self, size=1024):
        self._size = size
        self._buckets = [None] * size

    def getHashIndex(self, word):
        h = 0
        for ch in word:
            h = (h * 31 + ord(ch)) % self._size
        return h

    def addPositionForWord(self, word, position):
        idx = self.getHashIndex(word)
        if self._buckets[idx] is None:
            self._buckets[idx] = {}
        if word not in self._buckets[idx]:
            self._buckets[idx][word] = WordEntry(word)
        self._buckets[idx][word].addPosition(position)

    def getWordEntry(self, word):
        idx = self.getHashIndex(word)
        bucket = self._buckets[idx]
        if bucket is None:
            return None
        return bucket.get(word, None)

    def getWordEntries(self):
        entries = []
        for bucket in self._buckets:
            if bucket:
                entries.extend(bucket.values())
        return entries


In [14]:
class MySet:
    def __init__(self):
        self._items = []

    def addElement(self, element):
        if element not in self._items:
            self._items.append(element)

    def union(self, other):
        result = MySet()
        for item in self._items:
            result.addElement(item)
        for item in other._items:
            result.addElement(item)
        return result

    def intersection(self, other):
        result = MySet()
        for item in self._items:
            if item in other._items:
                result.addElement(item)
        return result

    def __iter__(self):
        return iter(self._items)

    def __len__(self):
        return len(self._items)


In [15]:
class PageIndex:
    def __init__(self):
        self._table = MyHashTable()

    def addPositionForWord(self, word, position):
        self._table.addPositionForWord(word, position)

    def getWordEntry(self, word):
        return self._table.getWordEntry(word)

    def getWordEntries(self):
        return self._table.getWordEntries()


class PageEntry:
    def __init__(self, page_name, pages_dir):
        self.pageName = page_name
        self._index = PageIndex()
        with open(f"{pages_dir}/{page_name}") as f:
            text = f.read()
        for word, pos in tokenize(text):
            self._index.addPositionForWord(word, Position(self, pos))

    def getPageIndex(self):
        return self._index


In [16]:
class InvertedPageIndex:
    def __init__(self):
        self._pages = {}
        self._global_table = MyHashTable(size=4096)

    def addPage(self, page_entry):
        self._pages[page_entry.pageName] = page_entry
        for word_entry in page_entry.getPageIndex().getWordEntries():
            for position in word_entry.getAllPositionsForThisWord():
                self._global_table.addPositionForWord(word_entry.word, position)

    def getPagesWhichContainWord(self, word):
        result = MySet()
        word = normalize(word.lower())
        word_entry = self._global_table.getWordEntry(word)
        if word_entry is None:
            return result
        for pos in word_entry.getAllPositionsForThisWord():
            result.addElement(pos.getPageEntry().pageName)
        return result

    def getPositionsOfWordInPage(self, word, page_name):
        word = normalize(word.lower())
        if page_name not in self._pages:
            return None, False
        word_entry = self._pages[page_name].getPageIndex().getWordEntry(word)
        if word_entry is None:
            return [], True
        positions = sorted(
            p.getWordIndex() for p in word_entry.getAllPositionsForThisWord()
            if p.getPageEntry().pageName == page_name
        )
        return positions, True


In [17]:
class SearchEngine:
    def __init__(self, pages_dir):
        self._index = InvertedPageIndex()
        self._pages_dir = pages_dir

    def performAction(self, action_message):
        parts = action_message.strip().split()
        action = parts[0]

        if action == "addPage":
            page_name = parts[1]
            page_entry = PageEntry(page_name, self._pages_dir)
            self._index.addPage(page_entry)
            print(f"Added page: {page_name}")

        elif action == "queryFindPagesWhichContainWord":
            word = parts[1]
            pages = self._index.getPagesWhichContainWord(word)
            if len(pages) == 0:
                print(f"No webpage contains word {word}")
            else:
                print(", ".join(sorted(pages)))

        elif action == "queryFindPositionsOfWordInAPage":
            word, page_name = parts[1], parts[2]
            positions, page_found = self._index.getPositionsOfWordInPage(word, page_name)
            if not page_found:
                print(f"No webpage {page_name} found")
            elif len(positions) == 0:
                print(f"Webpage {page_name} does not contain word {word}")
            else:
                print(", ".join(str(p) for p in positions))


In [18]:
PAGES_DIR    = r"C:\Projects\IITJ\MLDB\Assignment-4\Q2_webSearch\webpages"
ACTIONS_FILE = r"C:\Projects\IITJ\MLDB\Assignment-4\Q2_webSearch\actions.txt"

engine = SearchEngine(PAGES_DIR)

print("=" * 60)
with open(ACTIONS_FILE) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        print(f"> {line}")
        engine.performAction(line)
        print()


> addPage stack_datastructure_wiki
Added page: stack_datastructure_wiki

> queryFindPagesWhichContainWord delhi
No webpage contains word delhi

> queryFindPagesWhichContainWord stack
stack_datastructure_wiki

> queryFindPagesWhichContainWord wikipedia
stack_datastructure_wiki

> queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines

> queryFindPagesWhichContainWord allain
No webpage contains word allain

> addPage stack_cprogramming
Added page: stack_cprogramming

> queryFindPagesWhichContainWord allain
stack_cprogramming

> queryFindPagesWhichContainWord C
stack_cprogramming

> queryFindPagesWhichContainWord C++
stack_cprogramming

> addPage stack_oracle
Added page: stack_oracle

> queryFindPagesWhichContainWord jdk
stack_oracle

> addPage stackoverflow
Added page: stackoverflow

> queryFindPagesWhichContainWord function
stack_cprogramming, stack_datastructure_wiki, stackoverflow

> addPage stacklighting
Add

In [19]:
ANSWERS_FILE = r"C:\Projects\IITJ\MLDB\Assignment-4\Q2_webSearch\answers.txt"
print("Expected answers from answers.txt:")
with open(ANSWERS_FILE) as f:
    for line in f:
        print(" ", line.strip())


Expected answers from answers.txt:
  No webpage contains word delhi
  stack_datastructure_wiki
  stack_datastructure_wiki
  Webpage stack_datastructure_wiki does not contain word magazines
  No webpage contains word allain
  stack_cprogramming
  stack_cprogramming
  stack_cprogramming
  stack_oracle
  stack_cprogramming, stack_datastructure_wiki, stackoverflow
  stackmagazine


---
## Part 3: PageRank on Spark

The PageRank update rule is:

$$r^{(i)} = \frac{1-\beta}{n} \mathbf{1} + \beta M r^{(i-1)}$$

where $M_{ij} = 1/\text{deg}(i)$ if edge $i \to j$ exists, else $0$.  
We run 40 iterations with $\beta = 0.8$.


In [20]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark.sql import SparkSession
import numpy as np
from scipy.sparse import csr_matrix

spark = SparkSession.builder.master("local[*]").appName("PageRank").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

small_path = "C:/Projects/IITJ/MLDB/Assignment-4/Q3_pagerank_graphs/small.txt"
whole_path  = "C:/Projects/IITJ/MLDB/Assignment-4/Q3_pagerank_graphs/whole.txt"

def load_graph(path):
    edges = set()
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                edges.add((int(parts[0]), int(parts[1])))
    return sc.parallelize(list(edges))

def run_pagerank(edges_rdd, beta=0.8, iterations=40):
    edges = edges_rdd.collect()
    nodes = sorted(set(n for pair in edges for n in pair))
    n = len(nodes)
    node_idx = {node: i for i, node in enumerate(nodes)}

    out_degree = np.zeros(n)
    for src, dst in edges:
        out_degree[node_idx[src]] += 1

    rows, cols, data = [], [], []
    for src, dst in edges:
        s, d = node_idx[src], node_idx[dst]
        rows.append(d)
        cols.append(s)
        data.append(1.0 / out_degree[s])

    M = csr_matrix((data, (rows, cols)), shape=(n, n))

    r = np.full(n, 1.0 / n)
    for _ in range(iterations):
        r = (1 - beta) / n + beta * M.dot(r)

    return sc.parallelize([(nodes[i], float(r[i])) for i in range(n)]), n

edges_small = load_graph(small_path)
ranks_small, n_small = run_pagerank(edges_small)
top1 = max(ranks_small.collect(), key=lambda x: x[1])
print(f"Small graph: {n_small} nodes")
print(f"Top node ID: {top1[0]}   score: {top1[1]:.4f}   (expected ~0.036)")

edges_whole = load_graph(whole_path)
ranks_whole, n_whole = run_pagerank(edges_whole)
sorted_ranks = sorted(ranks_whole.collect(), key=lambda x: -x[1])

print(f"\nWhole graph: {n_whole} nodes")
print("\nTop 5 nodes by PageRank:")
for i, (node, score) in enumerate(sorted_ranks[:5], 1):
    print(f"  {i}. Node {node:4d}  score={score:.6f}")

print("\nBottom 5 nodes by PageRank:")
for i, (node, score) in enumerate(sorted_ranks[-5:], 1):
    print(f"  {i}. Node {node:4d}  score={score:.6f}")

Small graph: 100 nodes
Top node ID: 53   score: 0.0357   (expected ~0.036)

Whole graph: 1000 nodes

Top 5 nodes by PageRank:
  1. Node  263  score=0.002020
  2. Node  537  score=0.001943
  3. Node  965  score=0.001925
  4. Node  243  score=0.001853
  5. Node  285  score=0.001827

Bottom 5 nodes by PageRank:
  1. Node  408  score=0.000388
  2. Node  424  score=0.000355
  3. Node   62  score=0.000353
  4. Node   93  score=0.000351
  5. Node  558  score=0.000329
